# Speech to Text

##1️⃣ Dependencias y Modelos

In [ ]:
# @title ⚠️ Configuración Inicial {display-mode: "form"}

from pathlib import Path
import torch
from typing import Optional, Tuple
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

BASE_DIR = Path('/content')
google_drive = True # @param {type:"boolean"}

if google_drive:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/v2v') if google_drive else None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


MODELS = {}
def get_or_load_model( model_key: str, loader_fn, *args, **kwargs ):
    """
    Carga un modelo la primera vez y lo reutiliza en llamadas posteriores.
    Args:
        model_key (str): Identificador único del modelo.
        loader_fn (callable): Función que carga el modelo.
        *args, **kwargs: Argumentos que se pasan al loader_fn.
    """
    global MODELS
    if model_key not in MODELS:
        try:
            MODELS[model_key] = loader_fn(*args, **kwargs)
            print(f"✅ Modelo '{model_key}' cargado")
        except Exception as e:
            raise RuntimeError(f"⚠️ Error al cargar modelo '{model_key}': {e}")
    return MODELS[model_key]


def adjust_audio_duration(
    audio: torch.Tensor,
    target_duration_samples: int,
    sample_rate: int,
    min_tempo: float = 0.6
) -> torch.Tensor:
    """
    Ajusta la duración del audio usando cambio de tempo.
    Args:
        audio: Tensor de audio [channels, samples]
        target_duration_samples: Duración objetivo en muestras
        sample_rate: Frecuencia de muestreo
        min_tempo: Tempo mínimo permitido
    """
    if audio.shape[-1] == 0 or target_duration_samples <= 0: return audio

    tempo = audio.shape[-1] / target_duration_samples
    safe_tempo = max(tempo, min_tempo)

    try:
        effector = torchaudio.io.AudioEffector(f'atempo={safe_tempo}')
        adjusted = effector.apply(audio.T, sample_rate).T
        return adjusted[:, :target_duration_samples]
    except Exception as e:
        print(f"⚠️ Error al ajustar tempo: {e}")
        return audio[:, :target_duration_samples]

def clone_voice_segments(
    path: Path,
    texts: list[dict],
    language,
    audio_ref,
    tts_fn,
    only_indices: Optional[list[int]] = None,
    slowdown_range: Optional[Tuple[float, float]] = None,
    stretch_target_ratio: float = 1.0,
    **kwargs
) -> Tuple[list[Path], list[int], int]:

    seqs_src_dir = path.parent / 'Seqs_src'
    seqs_trg_dir = path.parent / 'Seqs_trg'
    seqs_trg_dir.mkdir(exist_ok=True)

    # Filtrar textos si se especifican índices
    texts_to_process = texts if only_indices is None else [texts[i-1] for i in only_indices]
    start_idx = 1 if only_indices is None else only_indices[0]

    times, path_es = [], []

    for idx, text_dict in enumerate(texts_to_process):
        text = text_dict['text']
        start_time = text_dict['start']

        # Número de secuencia
        seq_num = start_idx + idx if only_indices else texts.index(text_dict) + 1

        # Cargar audio original para comparar duración
        audio_en_path = seqs_src_dir / f'vocals_{seq_num:04d}.wav'
        if not audio_en_path.exists():
            print(f"⚠️ No se encontró: {audio_en_path}")
            continue

        arr_en, rate_en = torchaudio.load(str(audio_en_path))
        dur_en = arr_en.shape[-1] / rate_en

        # Generar audio con TTS
        arr, rate = tts_fn(text, language, audio_ref, **kwargs)

        dur_es = arr.shape[-1] / rate
        duration_ratio = dur_es / dur_en if dur_en > 0 else 0
        print(f"{idx:2d}. ", end='')

        # Ajustar duración según casos
        if duration_ratio > 1.0:
            # CASO 1: Audio más largo - acelerar
            print(f"Fast: {dur_es:5.2f}s → {dur_en:5.2f}s",  end=' ')
            target_samples = int(dur_en * rate)
            arr = adjust_audio_duration(arr, target_samples, rate)

        elif slowdown_range and slowdown_range[0] < duration_ratio < slowdown_range[1]:
            # CASO 2: Audio más corto en rango - estirar
            target_dur_sec = dur_es + (dur_en - dur_es) * stretch_target_ratio
            print(f"Slow: {dur_es:5.2f}s → {target_dur_sec:5.2f}s", end=' ')
            target_samples = int(target_dur_sec * rate)
            arr = adjust_audio_duration(arr, target_samples, rate)

        else: print(f"Stay: {dur_es:5.2f}s → {dur_en:5.2f}s", end=' ')
        print(f" {text}")

        # Guardar audio generado
        audio_es_path = seqs_trg_dir / f'{audio_en_path.stem}.wav'
        torchaudio.save(str(audio_es_path), arr, sample_rate=rate)
        path_es.append(audio_es_path)

        # Calcular tiempo de inicio en muestras
        start_samples = int(start_time * rate)
        times.append(start_samples)

    return path_es, times, rate


In [ ]:
# @title ⚠️ Gemini Flash 2.5 - Traductor LLM {display-mode: "form"}
from google import genai
from google.genai import types
import json
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

try:
    if not GOOGLE_API_KEY: raise ValueError("La variable de entorno GOOGLE_API_KEY no está configurada.")
except ValueError as e: print("Por favor, configura tu GOOGLE_API_KEY.")

LANGUAGE_NAMES = {
    "es": "Español",
    "en": "Inglés",
    "fr": "Francés",
    "de": "Alemán",
    "it": "Italiano",
    "pt": "Portugués",
    "ja": "Japonés",
    "zh-CN": "Chino",
}

def translate_text_list(
    list_sentences: list[str],
    target_lang: str,
    source_lang: str = "en",
) -> list[str] | None:
    """
    Traduce una lista de frases a un idioma objetivo usando Gemini,
    manteniendo la coherencia entre las frases.

    Args:
        list_sentences (list[str]): Lista de frases a traducir.
        target_language_code (str): Código del idioma objetivo (ej: "es", "fr").
        source_language_code (str): Código del idioma de origen (ej: "en", "es").

    Returns:
        list[str] | None: Lista de frases traducidas o None si ocurre un error.
    """
    if not list_sentences:
        return []

    if not GOOGLE_API_KEY: # Verifica si la API key fue cargada
        print("Error: La API Key de Google no está configurada. No se puede traducir.")
        return None

    client = genai.Client(api_key=GOOGLE_API_KEY)

    # Nombres de los idiomas para un prompt más claro
    source_lang_name = LANGUAGE_NAMES.get(source_lang, 'español')
    target_lang_name = LANGUAGE_NAMES.get(target_lang, 'ingles')

    prompt = f"""
Eres un traductor experto. Traduce el siguiente texto del {source_lang_name} al {target_lang_name}.
El texto original está compuesto por varias frases que forman un párrafo o una secuencia lógica.
Es CRUCIAL que mantengas la coherencia, el tono y el contexto entre todas las frases traducidas.
Utiliza las mejores prácticas de traducción para el {target_lang_name}.

Texto original en {source_lang_name}:
---
{json.dumps(list_sentences, indent=2, ensure_ascii=False)}
---

Por favor, devuelve la traducción como una LISTA JSON de strings, donde cada string
corresponda a una frase traducida, manteniendo el mismo orden que las frases originales.
Cada string traducida debe tener una longitud similar a la del lenguaje original.
Asegúrate de que la salida sea únicamente la lista JSON y nada más.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[prompt],
        config={'response_mime_type': 'application/json',}
        # generation_config=genai.types.GenerationConfig(temperature=0.2) # Opcional: para traducciones más deterministas
    )
    return json.loads(response.text.strip())


def generate_tiktok_title(
    youtube_url: str,
    target_lang: str = "es",
) -> str | None:

    client = genai.Client(api_key=GOOGLE_API_KEY)
    target_lang_name = LANGUAGE_NAMES.get(target_lang, 'español')

    prompt = f"""Eres un experto en marketing digital y creación de contenido viral.
Tu misión es generar EL TÍTULO MÁS LLAMATIVO para TikTok que despierte curiosidad
y haga que la gente quiera ver el video completo.

Requisitos:
- El título debe estar en español.
- Usa EMOJIS impactantes y relevantes que capten la atención.
- Incluye 5 HASHTAGS virales y relacionados con el tema del video.
- Hazlo breve, explosivo y con estilo auténtico de TikTok.
- El título debe provocar intriga, emoción o sorpresa.

IMPORTANTE: Devuelve **solo el título final**, sin explicaciones ni opciones adicionales.
"""
    model = "gemini-2.5-flash"
    contents = types.Content(
        parts=[
            types.Part( file_data=types.FileData(file_uri=youtube_url) ),
            types.Part( text=prompt )
        ]
    )
    response = client.models.generate_content(model = model, contents = contents)
    return response.text


In [ ]:
# @title ⚠️ YT-dlp / Demucs - Descarga y Separación de Audio {display-mode: "form"}
# %%capture

if 'yt_imported' not in globals():
    !uv pip install "yt-dlp[default]"
    !uv pip install lameenc openunmix diffq julius --no-deps
    yt_imported = True

import os, sys, subprocess
from yt_dlp import YoutubeDL

QUEUE_DIR = BASE_DIR / 'Queue'

# Clonar y configurar Demucs
if not Path('/content/demucs').exists():
    !git clone https://github.com/adefossez/demucs.git /content/demucs
    !gdown --fuzzy --folder https://drive.google.com/drive/folders/1T3M5K-67CQpCI6IC9d2xjga6r4QcjkFo?usp=drive_link -O /content/demucs/demucs

class OneLineLogger:
    def debug(self, msg):
        if msg.startswith('[download]'):
            print(end='\r' + msg, flush=True)
            if '100%' in msg: print()

    def info(self, msg): print(f"\nℹ️ {msg}")
    def warning(self, msg): print(f"\n⚠️ {msg}")
    def error(self, msg): pass


def download_video(url: str, output_dir: Path = QUEUE_DIR, resolution=None, language=None) -> tuple[Path, dict]:
    """
    Descarga un video de YouTube y extrae información.
    Args:
        url: URL del video de YouTube
        output_dir: Directorio de salida

    Returns: Tupla (path, info) con la ruta del archivo y la información del video
    """
    cookies_file = DRIVE_DIR / 'cookies.txt' if google_drive else None

    video = "bv[vcodec^=avc1]"
    video += f"[width<={resolution}]" if resolution else ''
    audio = f"ba[language^={language}]" if language else "ba"

    opts = {
        'outtmpl': str(output_dir / '%(title)s' / '%(title)s.%(ext)s'),
        'format': f"{video} + {audio}",
        'merge_output_format': 'mp4',
        'cookiefile' : str(cookies_file),
        'logger': OneLineLogger(),
        'js_runtimes': {'node': {}},
        'remote_components': ['ejs:github'],
        'quiet': True,
    }
    try:
        with YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)
            media = info['requested_downloads'][0]['filepath']
            print(
                f"✅ {Path(media).name}, {info.get('format', 'N/A')}, "
                f"{info.get('fps', 'N/A')}, {info.get('language', 'N/A')}, "
                f"{info.get('vcodec', 'N/A')}, {info.get('acodec', 'N/A')}"
            )

        return Path(media), info
    except Exception as e:
        print(f"❌ Error al descargar video: {e}")
        raise

def separate_vocals(video_path: Path, output_dir: Path = None) -> tuple[Path, Path]:
    """
    Separa las voces del audio usando Demucs.
    Args:
        video_path: Ruta al archivo de video
        output_dir: Directorio de salida (por defecto el mismo que el video)

    Returns: Tupla (vocals_path, no_vocals_path) con las rutas de los archivos generados
    """
    if output_dir is None: output_dir = video_path.parent

    vocals_path = output_dir / 'vocals.wav'
    no_vocals_path = output_dir / 'no_vocals.wav'

    # Si ya existen, no reprocesar
    if vocals_path.exists() and no_vocals_path.exists():
        print(f"✅ Audio de voz ya existe.")
        return vocals_path, no_vocals_path

    demucs_dir = Path('/content/demucs')
    original_dir = os.getcwd()

    try:
        os.chdir(demucs_dir)

        # Ejecutar separación de voces
        cmd = f'python -m demucs.separate "{video_path}" --two-stems vocals -o "{output_dir}"'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0: print(f"⚠️ Advertencia en separación: {result.stderr}")

        # Mover archivos generados
        demucs_output = output_dir / 'htdemucs' / video_path.stem
        if demucs_output.exists():
            vocals_temp = demucs_output / 'vocals.wav'
            no_vocals_temp = demucs_output / 'no_vocals.wav'

            if vocals_temp.exists(): vocals_temp.rename(vocals_path)
            if no_vocals_temp.exists(): no_vocals_temp.rename(no_vocals_path)

            # Limpiar directorio temporal
            subprocess.run(f'rm -rf "{demucs_output.parent}"', shell=True)

        return vocals_path, no_vocals_path

    finally: os.chdir(original_dir)

def yt(url: str, demucs: bool = True) -> tuple[Path, dict]:
    print(url)
    path, info = download_video(url)
    if demucs: separate_vocals(path)
    return path, info

In [ ]:
# @title ⚠️ Whisper - Transcripción STT {display-mode: "form"}
# %%capture

if 'whisper_imported' not in globals():
    !uv pip install -U git+https://github.com/jianfch/stable-ts.git --no-deps
    !uv pip install openai-whisper --no-deps
    whisper_imported = True

import stable_whisper
import torchaudio
import contextlib, io
from copy import deepcopy
from typing import Optional, Union, Tuple

WHISPER_DIR = DRIVE_DIR / 'whisper_turbo' if DRIVE_DIR else None
WHISPER_MODEL = get_or_load_model(
    "whisper",
    stable_whisper.load_model,
    'turbo',
    device = DEVICE,
    download_root=WHISPER_DIR
)

def transcribe_and_process(
    media_path = None, segments = None,
    gap: float = 1, duration: int = 7,
) -> Tuple[object, object]:

    try:
        # Si no se pasan segmentos, transcribir
        if segments is None:
            if media_path is None: raise ValueError("media_path / segments.")

            segments = WHISPER_MODEL.transcribe( str(media_path), verbose=None )

            _speaker_diarization, f_err = speaker_diarization, io.StringIO()

            with contextlib.redirect_stderr(f_err):
                diarize_segments = _speaker_diarization( DIARIZATION_MODEL, str(media_path) )

            result = process_audio( segments, diarize_segments,
                group_segments=True, max_duration=duration, max_gap=gap
            )

        return result, segments, diarize_segments

    except Exception as e:
        print(f"❌ Error en transcripción/procesamiento: {e}")
        raise

def write_subtitle_file(segments, output_path: Path, word_level=False):
    """
    Escribe archivo de subtítulos.
    Args:
        segments: Segmentos procesados
        output_path: Ruta de salida
        format: Formato ('srt' o 'vtt')
    """
    # Guarda referencia original ANTES de modificar
    _original_to_srt_vtt = segments.to_srt_vtt

    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        result = _original_to_srt_vtt(str(output_path), word_level=word_level)

    print(f"✅ Subtítulos guardados")


def run_main(media: Union[str, Path], url, translate: Union[bool, str] = False):
    media_path = Path(media)
    vocals_path = media_path.with_name('vocals.wav')

    if not vocals_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo de voces: {vocals_path}")

    # Transcribir y procesar
    result, transcription, diarization = transcribe_and_process(vocals_path)

    # Extraer datos en una sola pasada
    times, texts, speakers = [], [], set()
    add_time, add_text, add_speaker = times.append, texts.append, speakers.add

    for r in result:
        add_time((float(r['start']), float(r['end'])))
        add_text(r['text'])
        add_speaker(r['speaker'])

    # Lógica de traducción
    translated_result = result

    if translate:
        translated_texts = translate_text_list(
            texts,
            target_lang = translate,
            source_lang = transcription.language
        )

        if translated_texts:
            translated_result = [
                {'start': s, 'end': e, 'text': txt}
                for (s, e), txt in zip(times, translated_texts)
            ]
        else: print("⚠️ No se pudo traducir, usando texto original")

    # Generar nombre de salida limpio
    output_name = generate_tiktok_title(url, translate if translate else transcription.language)
    print(f"✅ {output_name}")

    return translated_result, output_name, result, transcription, diarization


In [ ]:
# %%capture
# @title ⚠️ Pyannote - Diarización {display-mode: "form"}
if 'pyannote_install' not in globals():
    !uv pip install git+https://github.com/pyannote/pyannote-audio.git --no-deps
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' sentencex
    !uv pip install 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps
    !uv pip install 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' 'torch-audiomentations>=0.12.0' torchcodec==0.7.0 'torchmetrics>=1.6.1' 'optuna>=4.2.0'
    pyannote_install = True

import re
import torch, torchaudio
from pyannote.audio import Pipeline
from pyannote.core import Segment
from sentencex import segment as sentencex_segment


DIARIZATION_DIR = DRIVE_DIR / 'speaker-diarization-community-1' if DRIVE_DIR else None
DIARIZATION_MODEL = get_or_load_model(
    "community-1",
    Pipeline.from_pretrained,
    DIARIZATION_DIR
)
DIARIZATION_MODEL.to(torch.device(DEVICE))


def speaker_diarization(diarization_model, wav_audio, num_speakers=None):
    waveform, sample_rate = torchaudio.load(wav_audio)

    output = diarization_model(
        { "waveform": waveform, "sample_rate": sample_rate },
        num_speakers=num_speakers
    )
    return output.exclusive_speaker_diarization

def assign_speakers_by_word(whisper_result, diarization):
    """
    Asigna hablantes a cada segmento de Whisper basándose en la intersección
    temporal de cada palabra con la diarización de Pyannote.
    """
    # Manejar si whisper_result es el objeto completo o solo la lista de segmentos
    segments_list = whisper_result.segments if hasattr(whisper_result, 'segments') else whisper_result

    assigned_segments = []

    for seg in segments_list:
        # Extraer datos (soporta tanto objeto como dict)
        seg_start = float(seg.start) if hasattr(seg, 'start') else seg['start']
        seg_end = float(seg.end) if hasattr(seg, 'end') else seg['end']
        seg_text = seg.text if hasattr(seg, 'text') else seg['text']
        seg_words = seg.words if hasattr(seg, 'words') else seg.get('words', [])

        # Fallback si no hay palabras
        if not seg_words: seg_words = [{'start': seg_start, 'end': seg_end, 'word': seg_text}]

        speaker_counts = {}
        enriched_words = []

        for word in seg_words:
            w_start = float(word['start'] if isinstance(word, dict) else word.start)
            w_end = float(word['end'] if isinstance(word, dict) else word.end)
            w_text = word['word'] if isinstance(word, dict) else word.word

            # Crear segmento de Pyannote para la palabra actual y busca intersección con la diarización
            word_segment = Segment(w_start, w_end)
            turn = diarization.crop(word_segment)
            best_speaker = "UNKNOWN"
            max_overlap = 0

            for segment, _, label in turn.itertracks(yield_label=True):
                overlap = segment & word_segment
                if overlap.duration > max_overlap:
                    max_overlap = overlap.duration
                    best_speaker = label

            if best_speaker != "UNKNOWN":
                speaker_counts[best_speaker] = speaker_counts.get(best_speaker, 0) + max_overlap

            enriched_words.append({
                "word": w_text, "start": w_start, "end": w_end, "speaker": best_speaker
            })

        # Gana el hablante con más tiempo acumulado en las palabras
        if speaker_counts: segment_speaker = max(speaker_counts, key=speaker_counts.get)
        else: segment_speaker = "UNKNOWN"

        assigned_segments.append({
            "start": seg_start,
            "end": seg_end,
            "text": seg_text,
            "speaker": segment_speaker,
            "words": enriched_words
        })

    return assigned_segments


def smart_group_segments_strict(assigned_segments, max_gap=1.5, max_duration=10):
    """
    Agrupa segmentos priorizando la integridad de la frase sobre la duración máxima.
    - NO corta en comas.
    - Si se excede la duración y no hay punto, sigue extendiendo hasta encontrar uno.
    """
    if not assigned_segments:
        return []

    # 1. APLANAR: Convertimos todo a una lista de palabras con sus metadatos
    all_words = []
    for seg in assigned_segments:
        seg_speaker = seg["speaker"]
        for w in seg["words"]:
            word_copy = w.copy()
            word_copy["speaker"] = seg_speaker
            all_words.append(word_copy)

    if not all_words:
        return []

    grouped_segments = []

    # Iniciamos el buffer con la primera palabra
    current_words = [all_words[0]]

    # SOLO signos fuertes. Quitamos comas (,) y punto y coma (;).
    strong_punct = re.compile(r'[.!?¡¿]')

    def create_segment_from_words(words_list):
        if not words_list: return None
        raw_text = " ".join([w["word"].strip() for w in words_list])
        clean_text = re.sub(r'\s+([.,!?:;])', r'\1', raw_text)

        return {
            "start": words_list[0]["start"],
            "end": words_list[-1]["end"],
            "text": clean_text,
            "speaker": words_list[0]["speaker"],
            "words": words_list
        }

    # 2. ITERAR PALABRA POR PALABRA
    for i in range(1, len(all_words)):
        next_word = all_words[i]

        # Manejo de buffer vacío (si acabamos de hacer un corte exacto)
        if not current_words:
            current_words.append(next_word)
            continue

        last_word = current_words[-1]

        # --- VERIFICACIONES DE CORTE DURO (Hard Cuts) ---
        gap = next_word["start"] - last_word["end"]
        speaker_changed = next_word["speaker"] != current_words[0]["speaker"]
        gap_too_big = gap > max_gap

        if speaker_changed or gap_too_big:
            # Aquí NO importa si hay punto o no, el audio se rompe físicamente (silencio o cambio de voz)
            grouped_segments.append(create_segment_from_words(current_words))
            current_words = [next_word]
            continue

        # --- VERIFICACIÓN DE DURACIÓN Y PUNTUACIÓN ---
        # Añadimos la palabra temporalmente para calcular tiempos
        current_words.append(next_word)

        current_duration = current_words[-1]["end"] - current_words[0]["start"]

        # Si nos pasamos del tiempo, intentamos buscar un punto de corte VÁLIDO
        if current_duration > max_duration:

            split_index = -1

            # Buscamos hacia atrás un signo FUERTE
            for idx in range(len(current_words) - 1, -1, -1):
                if strong_punct.search(current_words[idx]["word"]):
                    split_index = idx
                    break

            if split_index != -1:
                # ¡ENCONTRADO!
                # Cortamos en el signo fuerte.
                part_one = current_words[:split_index+1]
                part_two = current_words[split_index+1:] # El sobrante pasa al siguiente

                grouped_segments.append(create_segment_from_words(part_one))
                current_words = part_two # Puede quedar vacío si el punto era la última palabra
            else:
                # NO ENCONTRADO (El cambio clave):
                # No hacemos nada. Dejamos que 'current_words' siga creciendo en la siguiente iteración
                # hasta que aparezca un punto o ocurra un hard cut (gap/speaker).
                pass

        # Si no nos pasamos de tiempo, simplemente continuamos (la palabra ya se agregó arriba)

    # 3. GUARDAR EL REMANENTE FINAL
    if current_words:
        grouped_segments.append(create_segment_from_words(current_words))

    return grouped_segments


# --- Función Wrapper ---
def process_audio(whisper_result, diarization_result, group_segments=True,
    include_words=False, max_gap=1.5, max_duration=10
):
    assigned_data = assign_speakers_by_word(whisper_result, diarization_result)

    if group_segments:
        # Usamos la versión STRICT
        assigned_data = smart_group_segments_strict(
            assigned_data, max_gap=max_gap, max_duration=max_duration
        )

    if not include_words:
        for seg in assigned_data: seg.pop("words", None)

    return assigned_data


In [ ]:
# @title ⚠️ LMNT TTS {display-mode: "form"}
# %%capture

if 'lmnt_install' not in globals():
    import io, torchaudio, requests, json, time

    LMNT_INTERNAL_KEY = '1fdc497ee58b4172aa9a0b82a3e14054'
    LMNT_AUTH0_COOKIE_VALUE = 'TBfDey7UWDbK5XWVTmLDWX'
    MIXAPANEL_COOKIE_VALUE = '%7B%22distinct_id%22%3A%20%2219be4349d3c31f-0f2ee89d3d2dd3-4c657b58-1fa400-19be4349d3d41a%22%2C%22%24device_id%22%3A%20%2219be4349d3c31f-0f2ee89d3d2dd3-4c657b58-1fa400-19be4349d3d41a%22%2C%22%24initial_referrer%22%3A%20%22https%3A%2F%2Fapp.lmnt.com%2F%22%2C%22%24initial_referring_domain%22%3A%20%22app.lmnt.com%22%7D'
    url_lmnt = "https://api.lmnt.com/v1/ai/speech/bytes"

    cookies = {
        'lmnt-auth0': LMNT_AUTH0_COOKIE_VALUE,
        'mp_576dc537cb2781fc026e98b5c5527486_mixpanel': MIXAPANEL_COOKIE_VALUE,
    }

    headers = {
        'accept': '*/*',
        'accept-language': 'es,es-ES;q=0.9,en;q=0.8',
        'origin': 'https://app.lmnt.com',
        'priority': 'u=1, i',
        'referer': 'https://app.lmnt.com/',
        'sec-ch-ua': '"Not(A:Brand";v="8", "Chromium";v="144", "Microsoft Edge";v="144"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Windows"',
        'sec-fetch-dest': 'empty',
        'sec-fetch-mode': 'cors',
        'sec-fetch-site': 'same-site',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36',
        'x-api-key': LMNT_INTERNAL_KEY,
    }
    lmnt_install = True

def lmnt_tts(
        text,
        language,
        voice_id,
        format='wav',
        temperature=1.2,
        top_p=0.85,
        sample_rate=24000
    ):

    files = {
        'text': (None, text),
        'voice': (None, voice_id),
        'format': (None, format),
        'language': (None, language),
        'model': (None, "blizzard"),
        'sample_rate': (None, str(sample_rate)),
        'temperature': (None, str(temperature)),
        'top_p': (None, str(top_p)),
    }

    try:
        response = requests.post(url_lmnt, headers=headers, files=files, cookies=cookies)
        time.sleep(2)

        response.raise_for_status()
        waveform, sample_rate = torchaudio.load(io.BytesIO(response.content))
        return waveform, sample_rate

    except requests.exceptions.HTTPError as e:
        print(f"\nError HTTP: {e}")
        print(f"Código de estado: {response.status_code}")
        print(f"Respuesta del servidor: {response.text}")
        return None, None

    except Exception as e:
        print(f"\nOcurrió un error inesperado: {e}")
        return None, None


def model_tts(text, language, audio_ref, **kwargs):
    return lmnt_tts( text, language, audio_ref, **kwargs )


## 2️⃣ Lista de Urls a Descargar

In [ ]:
queue = '''

'''
queue = [url.strip() for url in queue.split('\n') if url.strip()]


# Configuración de traducción
translate = "None" # @param ["None","es", "pt", "en"]
if translate == "None": translate = False

##3️⃣ Procesar Videos

In [ ]:
# @title ⚠️ Descargar y Transcribir Videos

import pickle
from pathlib import Path
from typing import Union


def process_video_queue(urls: list[str], translate_lang: Union[bool, str] = False):
    """
    Procesa una cola de videos: descarga, transcribe y traduce.
    Args:
        urls: Lista de URLs de YouTube
        translate_lang: Idioma objetivo para traducción (False = no traducir)
    """
    results = []
    add_result = results.append

    for idx, u in enumerate(urls, 1):
        print(f"Procesando: {u} {idx}/{len(urls)}")

        try:
            # Descargar video y separar audio
            path, info = yt(u, demucs=True)

            # Transcribir y traducir
            translated_result, output_name, result, transcription, diarization = run_main(path, u, translate=translate_lang)

            # Guardar resultados
            output_file = path.with_name('result.pkl')
            with open(output_file, 'wb') as f: pickle.dump((translated_result, output_name, u, result, transcription, diarization), f)

            print(f"✅ Video procesado:{' '+translate_lang+',' if translate_lang else ''} {len(translated_result)} segmentos")
            print(f"{'='*60}\n")

            add_result({
                'path': path,
                'output_name': output_name,
                'url': u,
                'texts': translated_result,
                'src_lang': transcription.language,
                'result': result,
                'transcription': transcription,
                'diarization': diarization
            })

        except Exception as e:
            print(f"❌ Error procesando {u}: {e}\n")
            continue

    print(f"✅ Procesamiento completado: {len(results)}/{len(urls)} videos")
    return results

# Procesar cola
if queue: results = process_video_queue(queue, translate)

In [ ]:
# @title ⚠️ Previsualizar Textos
import pandas as pd

show_original = True # @param {type:"boolean"}
show_traduction = True # @param {type:"boolean"}
show_comparation = True # @param {type:"boolean"}

paths = []
add_path = paths.append
for r in results: add_path(r['path'])

for i, path in enumerate(paths, 1):
    with open(path.with_name('result.pkl'), 'rb') as f:
        texts, output_name, url, result, transcription, diarization = pickle.load(f)

    print(f'Id {i}: {path.stem}','\t')


    if show_original and not show_comparation:
        display(list(result))
        print()

    if show_traduction and not show_comparation:
        print(pd.DataFrame(texts).to_string())
        # display(texts)
        print()

    if show_comparation:
        for i, j in zip(result, texts):
            if show_original: print(i['text'])
            if show_traduction: print(j['text'])
            print()
    print()

In [ ]:
# @title ##⚠️ Dividir en Secuencias

def split_audio_sequences(
    audio_path: Path,
    texts: list[dict],
    output_dir: Path = None,
    edit_indices: Optional[list[int]] = None
) -> list[Path]:
    """
    Divide el audio en secuencias según los segmentos de texto.
    Args:
        audio_path: Ruta al archivo de audio
        texts: Lista de diccionarios con 'text', 'start', 'end'
        output_dir: Directorio de salida
        edit_indices: Índices de secuencias a procesar (None = todas)
    """
    if output_dir is None: output_dir = audio_path.parent / 'Seqs_src'
    output_dir.mkdir(exist_ok=True)

    arr, org_sr = torchaudio.load(str(audio_path))
    generated_files = []

    # Filtrar textos si se especifican índices
    texts_to_process = texts if edit_indices is None else [texts[i-1] for i in edit_indices]

    for idx, text_dict in enumerate(texts_to_process):
        start = int(text_dict['start'] * org_sr)
        end = int(text_dict['end'] * org_sr)
        arr_split = arr[:, start:end]

        # Calcular número de secuencia
        if edit_indices: seq_num = edit_indices[idx]
        else: seq_num = texts.index(text_dict) + 1

        audio_filename = f'{audio_path.stem}_{seq_num:04d}.wav'
        audio_path_out = output_dir / audio_filename

        torchaudio_save, f_err = torchaudio.save, io.StringIO()
        with contextlib.redirect_stderr(f_err):
            torchaudio_save(str(audio_path_out), arr_split, sample_rate=org_sr)

        generated_files.append(audio_path_out)

    return generated_files


def split_all_sequences(video_paths: list[Path]):
    """
    Divide el audio de todos los videos en secuencias.
    Args: video_paths: Lista de rutas a videos procesados
    """
    for path in video_paths:
        texts_file = path.with_name('result.pkl')

        if not texts_file.exists():
            print(f"⚠️ Saltando {path.stem}: no hay archivo de textos")
            continue

        with open(texts_file, 'rb') as f:
            texts, output_name, url, result, transcription, diarization = pickle.load(f)

        split_audio_sequences(
            audio_path=path.with_name('vocals.wav'),
            texts=texts,
            output_dir=path.parent / 'Seqs_src'
        )
        print(f"✅ Secuencias: {path.stem}")

if paths: split_all_sequences(paths)
else: print("⚠️ No hay videos para procesar")


# Text to Speech

In [ ]:
# @title ## 6️⃣ Seleccionar Speaker y Renderizar Videos
"""Genera audio clonado y renderiza videos finales"""
# %%capture

import pickle
import subprocess
from IPython.display import display

TEMP_DIR = BASE_DIR / 'temp'
TEMP_DIR.mkdir(exist_ok=True)


# Configuración de speaker
mrbeast_wav = 'mr_beast.wav'
speaker = "mrbeast_lmnt" # @param ["mrbeast_lmnt", "Dionisio Schuyler", "mrbeast_wav", "en-US-AndrewMultilingualNeural", "Chatterbox"]

# Determinar speaker real
if speaker == "mrbeast_wav": speaker_audio = mrbeast_wav
elif speaker == "mrbeast_lmnt": speaker_audio = "d47094be-9814-4323-9000-e735e04d8732"
else: speaker_audio = speaker

def render_video(
    video_path: Path,
    output_name: str,
    vocals_es_path: Path,
    no_vocals_path: Path,
    output_dir: Path = TEMP_DIR
) -> Path:
    """
    Renderiza el video final combinando video original con audio clonado.
    Args:
        video_path: Ruta al video original
        output_name: Nombre del archivo de salida
        vocals_es_path: Ruta al audio de voces clonadas
        no_vocals_path: Ruta al audio sin voces
        output_dir: Directorio de salida

    Returns: Ruta al video renderizado
    """
    output_path = output_dir / f'{output_name}.mp4'

    cmd = [
        'ffmpeg', '-y', '-v', 'error',
        '-i', str(video_path),
        '-i', str(vocals_es_path),
        '-i', str(no_vocals_path),
        '-filter_complex',
        '[0]volume=0.00[a0];[1]volume=3.5[a1];[2]volume=1.0[a2];[a0][a1][a2]amix=inputs=3[a]',
        '-map', '0:v',
        '-map', '[a]',
        '-c:v', 'copy',
        str(output_path)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0: return output_path
    else:
        print(f"❌ Error al renderizar: {result.stderr}")
        raise RuntimeError(f"Error en ffmpeg: {result.stderr}")

def merge_cloned_audio(
    original_audio_path: Path,
    cloned_segments: list[Path],
    segment_times: list[int],
    sample_rate: int
) -> torch.Tensor:
    """
    Combina segmentos de audio clonados en un solo archivo.
    Args:
        original_audio_path: Ruta al audio original
        cloned_segments: Lista de rutas a segmentos clonados
        segment_times: Tiempos de inicio de cada segmento (en muestras)
        sample_rate: Frecuencia de muestreo
    Returns: Tensor de audio combinado
    """
    arr, org_sr = torchaudio.load(str(original_audio_path))
    fix_es = torchaudio.functional.resample(arr, orig_freq=org_sr, new_freq=sample_rate)
    fix_es[:, :] = 0

    for start_samples, audio_seg_path in zip(segment_times, cloned_segments):
        arr_es, _ = torchaudio.load(str(audio_seg_path))
        end_samples = start_samples + arr_es.shape[1]
        if end_samples <= fix_es.shape[1]:
            fix_es[:, start_samples:end_samples] = arr_es

    return fix_es


# Procesar todos los videos
rendered_videos = []

for i, pt in enumerate(paths, 1):
    texts_file = pt.with_name('result.pkl')

    if not texts_file.exists():
        print(f"⚠️ Saltando {pt.stem}: no hay archivo de textos")
        continue

    with open(texts_file, 'rb') as f:
        texts, output_name, url, result, transcription, diarization = pickle.load(f)

    print(f"Id {i}:\t{pt.stem}")
    print(f"{output_name}")
    try:
        # Clonar voz
        paths_es, times, freq = clone_voice_segments(
            pt,
            texts,
            translate if translate else transcription.language,
            speaker_audio,
            model_tts,
            slowdown_range=(0.5, 0.8),
            stretch_target_ratio=0.85
        )

        # Combinar audio clonado
        vocals_es_path = pt.with_name('vocals_es.wav')
        merged_audio = merge_cloned_audio(
            pt.with_name('vocals.wav'),
            paths_es,
            times,
            freq
        )
        torchaudio.save(str(vocals_es_path), merged_audio, freq)

        # Renderizar video
        no_vocals_path = pt.with_name('no_vocals.wav')
        if no_vocals_path.exists():
            output_video = render_video(
                pt,
                output_name,
                vocals_es_path,
                no_vocals_path
            )
            rendered_videos.append(output_video)

        else: print(f"⚠️ No se encontró audio sin voces: {no_vocals_path}")

    except Exception as e:
        print(f"❌ Error procesando {pt.stem}: {e}")
        continue
    print()

print(f"\n✅ Renderizado completado: {len(rendered_videos)} videos")

In [ ]:
# @markdown ## 7️⃣ Vista Previa de Video Renderizado
"""Muestra vista previa del video final renderizado"""

import subprocess
import json
import pickle
import sys
import time
import threading
from google.colab import files
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
import shutil


def upload_to_catbox(file_path: Path) -> str:
    cmd = [
        'curl', '-F', 'reqtype=fileupload',
        '-F', 'time=24h',
        '-F', f'fileToUpload=@{file_path}',
        'https://litterbox.catbox.moe/resources/internals/api.php'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0: return result.stdout.strip()
    else:
        print(f"⚠️ Error al subir: {result.stderr}")
        return ""

def get_video_metadata(video_path: Path) -> dict:
    cmd = [
        'ffprobe', '-v', 'quiet',
        '-print_format', 'json',
        '-show_format', '-show_streams',
        str(video_path)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    return json.loads(result.stdout)

def create_low_res_preview(input_path: Path, output_path: Path = None) -> Path:
    if output_path is None:
        output_path = input_path.parent / 'low_preview.mp4'

    # Extraer metadatos del primer stream de video
    # meta = get_video_metadata(input_path)
    # streams = meta.get("streams", [])
    # video_stream = next((s for s in streams if s.get("codec_type") == "video"), {})
    # pix_fmt = video_stream.get("pix_fmt", "yuv420p")
    # color_space = video_stream.get("color_space", "bt709")
    # color_primaries = video_stream.get("color_primaries", "bt709")
    # color_trc = video_stream.get("color_transfer", "bt709")

    # cmd = [
    #     'ffmpeg', '-y', '-v', 'error',
    #     '-i', str(input_path),
    #     '-vf', f'scale=360:-1,format={pix_fmt}',
    #     '-c:a', 'copy',
    #     '-vcodec', 'h264_nvenc',
    #     '-pix_fmt', pix_fmt,
    #     '-colorspace', color_space,
    #     '-color_primaries', color_primaries,
    #     '-color_trc', color_trc,
    #     '-b:v', '1000k',
    #     str(output_path)
    # ]

    cmd = [
        'ffmpeg', '-y', '-v', 'error',
        '-i', str(input_path),
        '-vf', 'scale=360:-1',
        '-c:a', 'copy',
        '-vcodec', 'h264_nvenc',
        # '-b:v', '1000k',
        str(output_path)
    ]

    subprocess.run(cmd, check=True)
    return output_path


# Seleccionar video para preview
play_id = 4 # @param {type:"integer"}
low_preview = False # @param {type:"boolean"}

if not paths or play_id < 1 or play_id > len(paths):
    print(f"⚠️ ID inválido. Videos disponibles: {len(paths)}")
else:
    path = paths[play_id - 1]

    # Cargar textos
    with open(path.with_name('result.pkl'), 'rb') as f:
        texts, output_name, url, result, transcription, diarization = pickle.load(f)

    output_path = TEMP_DIR / f'{output_name}.mp4'

    if not output_path.exists(): print(f"⚠️ Video renderizado no encontrado: {output_path}")
    else:
        # Obtener metadatos
        metadata = get_video_metadata(path)
        video_stream = next(
            (s for s in metadata.get('streams', []) if s.get('codec_type') == 'video'),
            {}
        )

        width = video_stream.get('width', 'N/A')
        height = video_stream.get('height', 'N/A')
        bitrate = video_stream.get('bit_rate', 'N/A')

        print(f"{output_path.name.strip()}")
        print(f"Resolución: {width}x{height}")
        print()

        # Mostrar textos
        print("📝 Segmentos de texto:")
        for idx, text_dict in enumerate(texts, 1):
            print(f"  {idx:2d}. {text_dict['text']}")
        print()

        # Vista previa
        if low_preview:
            low_path = create_low_res_preview(output_path)
            print("📤 Vista previa SD:", end=' ')
            low_url = upload_to_catbox(low_path)
            if low_url: print(f"{low_url}\n")



        # Variable global para el título
        new_name = output_name

        def spinner(text="📤 Generando Título..."):
            chars = ["|", "/", "-", "\\"]
            i = 0
            while not stop_spinner:
                sys.stdout.write(f"\r{text} {chars[i % len(chars)]}")
                sys.stdout.flush()
                i += 1
                time.sleep(0.1)

        def run_title_generation():
            global new_name, stop_spinner
            stop_spinner = False
            t = threading.Thread(target=spinner)
            t.start()

            # Generar título inicial o regenerado
            new_name = generate_tiktok_title(url, translate if translate else transcription.language)

            stop_spinner = True
            t.join()
            sys.stdout.write(f"\r{new_name}\n")

        # --- Primera ejecución automática ---
        # run_title_generation()

        # --- Botón para regenerar título ---
        def regenerate_title(b): run_title_generation()

        regen_button = widgets.Button(description="🔄 Regenerar título")
        regen_button.on_click(regenerate_title)

        # --- Botón para descargar video ---
        def download_video(b):
            video_path = TEMP_DIR / f'{new_name}.mp4'
            if output_path != video_path: shutil.copy(output_path, video_path)
            print()
            files.download(video_path)

        download_button = widgets.Button(description="📤 Descargar video")
        download_button.on_click(download_video)

        buttons_box = widgets.HBox([regen_button, download_button])
        display(buttons_box)




In [ ]:
import os
import zipfile
from google.colab import files

def comprimir_carpeta(carpeta_origen, archivo_zip_destino):
    # Crear archivo ZIP
    with zipfile.ZipFile(archivo_zip_destino, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for raiz, _, archivos in os.walk(carpeta_origen):
            for archivo in archivos:
                ruta_completa = os.path.join(raiz, archivo)
                ruta_relativa = os.path.relpath(ruta_completa, carpeta_origen)
                zipf.write(ruta_completa, ruta_relativa)

    # Descargar el archivo ZIP
    files.download(archivo_zip_destino)

comprimir_carpeta(TEMP_DIR, str(TEMP_DIR)+'.zip')

###⚠️Text Edit

In [ ]:
# @title ### ✅ Editar y Regenerar Textos
"""Edita textos y regenera solo los segmentos modificados"""

import pickle
from IPython.display import Audio, display

def find_changed_segments(original_texts: list[dict], edited_texts: list[dict]) -> list[int]:
    """
    Encuentra los índices de los segmentos que han cambiado.
    Args:
        original_texts: Lista original de textos
        edited_texts: Lista editada de textos

    Returns: Lista de índices (1-indexed) de segmentos modificados
    """
    changed = []
    for idx, (orig, edit) in enumerate(zip(original_texts, edited_texts), 1):
        if str(orig) != str(edit): changed.append(idx)

    return changed


def parse_sequence_input(seq_input: str, total_segments: int) -> list[int]:
    """
    Parsea la entrada de secuencias a una lista de índices.
    Args:
        seq_input: String con secuencias separadas por comas, guiones para rangos, o ':' para todas
        total_segments: Número total de segmentos disponibles

    Returns: Lista de índices de secuencias (1-indexed)
    """
    if not seq_input or not seq_input.strip():
        raise ValueError("La entrada está vacía. Usa números separados por comas (ej: 1,3,5), '-' para rangos (ej: 2-4) o ':' para todas.")

    seq_input = seq_input.strip()

    # Si es ':', devolver todas las secuencias
    if seq_input == ':': return list(range(1, total_segments + 1))

    indices: list[int] = []
    for part in seq_input.split(','):
        part = part.strip()
        if not part: continue

        if '-' in part:
            try: start, end = map(int, part.split('-'))
            except ValueError:
                raise ValueError(f"Formato inválido en rango: '{part}'. Usa números enteros (ej: 2-5).")
            if start > end:
                raise ValueError(f"Rango inválido: {part}. El inicio debe ser menor o igual al fin.")
            indices.extend(range(start, end + 1))
        else:
            try: indices.append(int(part))
            except ValueError:
                raise ValueError(f"Formato inválido: '{part}'. Usa números separados por comas (ej: 1,3,5).")

    # Validar que los índices estén en rango
    invalid = [idx for idx in indices if idx < 1 or idx > total_segments]
    if invalid:
        raise ValueError(
            f"Índices fuera de rango: {invalid}. "
            f"Rango válido: 1-{total_segments}"
        )

    # Eliminar duplicados y ordenar
    return sorted(set(indices))


def regenerate_segments(
    video_path: Path,
    texts: list[dict],
    speaker_audio: Union[str, Path],
    changed_indices: list[int],
    language = None,
    slowdown_range: tuple = (0.5, 0.80),
    stretch_target_ratio: float = 0.85,
    **kwargs
):
    """
    Regenera solo los segmentos de audio que han cambiado.
    Args:
        video_path: Ruta al video
        texts: Lista de textos (puede incluir ediciones)
        speaker_audio: Audio de referencia del speaker
        changed_indices: Índices de segmentos a regenerar (1-indexed)
        language: Código de idioma
        slowdown_range: Rango para estirar audio
        stretch_target_ratio: Ratio objetivo para estirar

    Returns: Lista de rutas a audios regenerados
    """
    if not changed_indices:
        print("✅ No hay cambios para regenerar")
        return []

    print(f"🔄 Regenerando {len(changed_indices)} segmento(s): {changed_indices}")

    # Dividir secuencias de audio originales primero
    split_audio_sequences(
        audio_path=video_path.with_name('vocals.wav'),
        texts=texts,
        edit_indices=changed_indices
    )

    # Regenerar solo los segmentos modificados
    paths, times, freq = clone_voice_segments(
        video_path,
        texts,
        language,
        speaker_audio,
        model_tts,
        only_indices=changed_indices,
        slowdown_range=slowdown_range,
        stretch_target_ratio=stretch_target_ratio,
        **kwargs
    )

    # Mostrar previews
    for idx, (seq_num, audio_path) in enumerate(zip(changed_indices, paths)):
        print(f"\n🎵 Secuencia {seq_num}:")
        if idx == 0: display(Audio(str(audio_path), autoplay=True))
        else: display(Audio(str(audio_path)))
        print()

    return paths

# Cargar textos actuales
if 'path' in locals() and path.exists():
    with open(path.with_name('result.pkl'), 'rb') as f:
        texts, output_name, url, result, transcription, diarization = pickle.load(f)
    display(texts)

else:
    print("⚠️ No hay video seleccionado. Ejecuta la celda de preview primero.")
    texts = []

In [ ]:
# @title ✅ Edita los textos aquí
"""Define textos editados aquí."""

# Ejemplo de textos editados (cambia según tus necesidades)
texts_edited =[{'start': 0.02,
  'end': 4.72,
  'text': 'Esta mujer tiene un talento increíble y gracias a él, gana miles de dólares.',
  'speaker': 'SPEAKER_00'},
 {'start': 6.38,
  'end': 12.08,
  'text': 'Coloca trozos de troncos en un molde de plástico y decora los bordes con diversas plantas y flores.',
  'speaker': 'SPEAKER_00'},
 {'start': 12.48,
  'end': 20.1,
  'text': 'Luego, retira el molde usando una solución especial y para que quede limpio y liso, lo pule con cera.',
  'speaker': 'SPEAKER_00'},
 {'start': 20.48,
  'end': 27.52,
  'text': 'Tras los toques finales, logra crear mesas y relojes que puede vender por miles. ¿Y tú? ¿Qué puntuación le darías a estos diseños?',
  'speaker': 'SPEAKER_00'}]

# Regenerar segmentos modificados
if 'texts' in locals() and texts and 'path' in locals():
    changed_indices = find_changed_segments(texts, texts_edited)

    if changed_indices:

        regenerated_paths = regenerate_segments(
            path,
            texts_edited,
            speaker_audio,
            changed_indices,
            language=translate if translate else transcription.language,
        )

        # Actualizar textos
        texts = texts_edited
        with open(path.with_name('result.pkl'), "wb") as f: pickle.dump((texts, output_name, url, result, transcription, diarization), f)

        print(f"✅{' Secuencia actualizada' if len(regenerated_paths) == 1 else 'Secuencias actualizadas'}")

    else:
        print("✅ No hay cambios detectados")
        texts = texts_edited
else:
    print("⚠️ Ejecuta primero la celda anterior para cargar los textos")

In [ ]:
# @title ### 🔄 Regenerar Secuencias Específicas
"""Regenera y previsualiza secuencias específicas de audio"""

# Entrada de secuencias a regenerar
seq_edit = "1" # @param {type:"string"}

tts_kwargs = False # @param {"type":"boolean"}
temperature = 2 # @param {"type":"number"}
length_penalty = None # @param {"type":"number"}
repetition_penalty = None # @param {"type":"number"}
top_k = None # @param {"type":"integer"}
top_p = 1 # @param {"type":"number"}

# Verificar que tenemos los datos necesarios
if 'path' not in locals() or not path.exists():
    print("⚠️ Error: No hay video seleccionado. Ejecuta la celda de preview primero.")
elif 'texts' not in locals() or not texts:
    print("⚠️ Error: No hay textos cargados. Ejecuta la celda de preview primero.")
else:
    try:
        # Parsear entrada
        sequence_indices = parse_sequence_input(seq_edit, len(texts))

        # Idioma
        language = translate if translate else transcription.language

        # Construir kwargs dinámicamente
        kwargs = {}
        if tts_kwargs:
            # Diccionario base con todos los parámetros
            params = {
                "temperature": temperature,
                "length_penalty": length_penalty,
                "repetition_penalty": repetition_penalty,
                "top_k": top_k,
                "top_p": top_p,
            }
            # Filtrar solo los que no sean -1
            kwargs = {k: v for k, v in params.items() if v != None}

        # Regenerar y previsualizar
        regenerated_paths = regenerate_segments(
            path,
            texts,
            speaker_audio,
            sequence_indices,
            language=translate if translate else transcription.language,
            **kwargs
        )
        print(f"✅{' Secuencia actualizada' if len(regenerated_paths) == 1 else 'Secuencias actualizadas'}")


    except ValueError as e:
        print(f"❌ Error de entrada: {e}")
    except Exception as e:
        print(f"❌ Error inesperado: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# @title ## ✅ Renderizar Video Final (con Ediciones)
"""Combina todos los segmentos editados y renderiza el video final"""

from IPython.display import display, HTML

low_video = False # @param {type:"boolean"}

if 'path' in locals() and 'texts' in locals() and 'output_name' in locals():
    print(path.stem)

    # Cargar todos los segmentos generados
    seqs_es_dir = path.parent / 'Seqs_trg'
    path_es = sorted(seqs_es_dir.glob("*.wav"))

    if not path_es: print("⚠️ No se encontraron segmentos generados. Ejecuta la regeneración primero.")
    else:
        # Obtener frecuencia de muestreo del primer segmento
        _, freq = torchaudio.load(str(path_es[0]))

        # Calcular tiempos de inicio
        times = [int(text_dict['start'] * freq) for text_dict in texts]

        # Combinar audio
        arr, org_sr = torchaudio.load(str(path.with_name('vocals.wav')))
        fix_es = torchaudio.functional.resample(arr, orig_freq=org_sr, new_freq=freq)
        fix_es[:, :] = 0

        for start_samples, audio_es_path in zip(times, path_es):
            arr_es, _ = torchaudio.load(str(audio_es_path))
            end_samples = start_samples + arr_es.shape[1]
            if end_samples <= fix_es.shape[1]:
                fix_es[:, start_samples:end_samples] = arr_es

        # Guardar audio combinado
        vocals_es_path = path.with_name('vocals_es.wav')
        torchaudio.save(str(vocals_es_path), fix_es, freq)

        # Renderizar video
        no_vocals_path = path.with_name('no_vocals.wav')
        if no_vocals_path.exists():
            output_video = render_video(
                path,
                output_name,
                vocals_es_path,
                no_vocals_path
            )

            # Vista previa
            if low_video:
                low_path = create_low_res_preview(output_video)
                print("📤 Subiendo versión de baja resolución...")
                low_url = upload_to_catbox(low_path)
                if low_url: print(f"✅ Vista previa baja res: {low_url}")

            else: display(buttons_box)

        else:
            print(f"⚠️ No se encontró audio sin voces: {no_vocals_path}")

else: print("⚠️ Ejecuta primero las celdas anteriores para cargar path, texts y output_name")